# 🧠 TRIBE v2 Playground: Colab launcher
1. **Runtime → Change runtime type → GPU** (T4 works; A100 is faster).
2. Add a Colab secret (🔑 sidebar) named `HF_TOKEN`: a Hugging Face *read* token (access to `meta-llama/Llama-3.2-3B` is optional: the app can use an ungated mirror of the same weights).
3. If this repo is private, add a secret `GH_TOKEN` (GitHub fine-grained token with read access to the repo).
4. Run all cells and open the `trycloudflare.com` link printed by the last cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
try:
    gh = userdata.get('GH_TOKEN'); auth = f'{gh}@'
except Exception:
    auth = ''
if not os.path.exists('nmai-capstone'):
    !git clone -q -b claude/pensive-faraday-gy9cjo https://{auth}github.com/gtougs/nmai-capstone.git
%cd nmai-capstone
!pip install -q uv -r requirements.txt "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2"

In [ ]:
from huggingface_hub import login, auth_check
login(token=userdata.get('HF_TOKEN'))
try:
    auth_check('meta-llama/Llama-3.2-3B'); print('✅ Official Llama 3.2 access OK')
except Exception:
    print('⚠️ No official Llama 3.2 access yet. In the app sidebar set "Text feature model" to "Ungated mirror (unsloth)". '
          'Note that video and audio need it too, because their speech is transcribed.')

In [ ]:
# Launch Streamlit + a free Cloudflare quick tunnel (no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, time
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true', '--server.port', '8501'])
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if (m := re.search(r'https://[\w-]+\.trycloudflare\.com', line)):
        print('🧠 Open the playground:', m.group(0)); break